# 09 — Rapport de qualité des données

**Rôle :** synthèse finale — couverture, champs garantis sans manquant, frontière de complétude (électronique vs backlog scanné), distributions clés. C'est le document de référence pour interpréter le futur backtest.

**Prérequis :** notebook 07 exécuté.

**Cellule 1 — Imports & chargement de la table canonique.**

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() or (p / ".git").exists():
            return p
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED    = PROJECT_ROOT / "data" / "processed"
REPORTS      = PROJECT_ROOT / "reports"
df = pd.read_csv(PROCESSED / "congress_transactions.csv", parse_dates=["transaction_date", "disclosure_date"])
print("Table canonique :", len(df), "transactions")

**Cellule 2 — Couverture par chambre et par an.**

In [ ]:
cov = (df.groupby([df["source"], df["disclosure_date"].dt.year]).size()
         .unstack(0).fillna(0).astype(int))
cov

**Cellule 3 — Contrôle zéro-manquant** sur les champs garantis.

In [ ]:
GUARANTEED = ["declarant_name", "chamber", "transaction_date", "disclosure_date",
              "ticker", "operation_type", "amount_midpoint"]
missing = {c: int(df[c].isna().sum()) for c in GUARANTEED}
missing

**Cellule 4 — Distributions clés.** Achats vs ventes, et statistiques de montant (utile pour la future règle de sortie).

In [ ]:
op = df["operation_type"].astype(str).str.lower()
buys  = int(op.str.startswith("p").sum())
sells = int(op.str.startswith("s").sum())
print("Achats:", buys, "| Ventes:", sells)
print(df["amount_midpoint"].describe())

**Cellule 5 — Rédaction du rapport de qualité.**

In [ ]:
lines = ["# Rapport de qualité — congress_transactions", "",
         f"Généré : {datetime.now(timezone.utc).isoformat(timespec='seconds')}", "",
         f"- Transactions : {len(df)}",
         f"- Déclarants uniques : {df['declarant_name'].nunique()}",
         f"- Achats / Ventes : {buys} / {sells}", "",
         "## Couverture (chambre x an)", "```", cov.to_string(), "```", "",
         "## Champs garantis — valeurs manquantes", "```", str(missing), "```", "",
         "## Frontière de complétude",
         "- Ère électronique, niveau Member.",
         "- Backlog : PDF House `no_text` (cf. 03) + Sénat papier/scanné < 2015 (cf. 05).",
         "- Commissions : Congrès actuel uniquement (cf. 01)."]
(REPORTS / "data_quality.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

Pipeline données de la semaine 1 terminé ✅.

Hors-scope (rappel) : OCR du backlog · mapping GICS→ETF (S2) · stratégie & backtest (S3-S4).